In [2]:
#Import the necessary libraries
using Random #use random variables
using HDF5 #manage files
using JLD #manage files
using Gadfly #plotting library
using LaTeXStrings #LaTeX symbols in graphs
using FFTW #Fast Fourier Transforms
using FastGaussQuadrature #Gaussian quadratures
using LinearAlgebra #Linear Algebra operations
using StatsBase
using Statistics
using SpecialFunctions
using Distributions
using Dierckx
using DSP
using DataFrames, GLM
#using Optim

# Estimation of the autocorrelation

In [3]:
#preliminaries
function estλ(spikes)
    T=spikes[end]-spikes[1]
    λ=length(spikes)/T
    return λ
end

function squ_error(f1,f)
    N=length(f1)
    SE=(1/N)*sum((abs.(f1-f).^2))
    return SE
end

function sum_bins(ord_spikes,nh,resolution,J)
    #J=sum(ord_spikes)
    a=0;res=resolution;A=length(ord_spikes);nh=Integer(nh)
    while (a+1)*nh<A
        temp_sum=sum(ord_spikes[a*nh+1:(a+1)*nh])
        ord_spikes[a*nh+1:(a+1)*nh].=1/(J*nh*res)*temp_sum
        a+=1
    end
    if sum(ord_spikes[a*nh+1:end])>0
        temp_sum=sum(ord_spikes[a*nh+1:end])
        ord_spikes[a*nh+1:end].=(1/(J*(A-a*nh)*res)*(A-a*nh)*temp_sum+nh*ord_spikes[(a)*nh])/(A-a*nh+nh)
    else
        ord_spikes[a*nh+1:end].=1/(J*nh*res)*temp_sum
    end
    return ord_spikes
end

function binned_autocorrelation(spikes,Num_spikes,resolution,Tmax)#This function orders the autocorrelation data in the smallest bins
    h=resolution;support=[x for x in 0:h:Tmax];bin_edges=support.+h/2;τn=length(support);g=zeros(τn);
    λ=estλ(spikes[1:Num_spikes])
    for n in 1:Num_spikes
        tn=spikes[n]
        Jn=searchsortedlast(spikes,tn+Tmax+h/2)
        temp_spikes=spikes[n:Jn].-tn
        k=Jn-n+1
        for m in 1:k
            tm=temp_spikes[m]
            u=searchsortedfirst(bin_edges,tm)
            g[u]+=1
        end
    end
    
    g[1]-=Num_spikes
    g[1]*=2
    return support,g,λ
end

function spline_autocorrelation(g,Num_spikes,resolution,bandwidth,Tmax,λ) #This function creates histograms with longer bins by summing the bins from a smaller histogram
    h=resolution;support=[x for x in 0:h:Tmax];bin_edges=support.+h/2;τn=length(support);
    hist=fill(Float64[],1);
    x=[1,2,3,4,5];y=[1,2,3,4,5];line_hist=Spline1D(x, y);quad_hist=Spline1D(x, y);cube_hist=Spline1D(x, y) #Create storage variables for the splines

    nh=bandwidth;Num_steps=Integer(floor(Tmax/(nh*h))); #Counting the number of histogram mid-points
    ord_spikes=[x for x in g]
    temp_hist=(sum_bins(ord_spikes,nh,h,Num_spikes).-λ)#Matching the theoretical autocorrelation
    hist=[temp_hist[j] for j in 1:τn] #Saving the bar histogram
        
        
        
    xs=[h*(nh*(j-0.5)-0.5) for j in 1:Num_steps] #Creating the new histogram mid-poins
    us=[searchsortedlast(support,u) for u in xs]
    ys=[temp_hist[u] for u in us]
        if length(xs)>1
                
            spline_hist1=Spline1D(xs,ys;k=1,bc="extrapolate") #First order splines
            line_hist=spline_hist1
        else
            line_hist=temp_hist
        end
            
        if length(xs)>2
                
            spline_hist2=Spline1D(xs,ys;k=2,bc="extrapolate") #Second order splines
            quad_hist=spline_hist2
        else
            quad_hist=temp_hist
        end
            
        if length(xs)>3
                
            spline_hist3=Spline1D(xs,ys;k=3,bc="extrapolate") #Third order splines
            cube_hist=spline_hist3
        else
            cube_hist=temp_hist
        end
        
        return hist,line_hist,quad_hist,cube_hist
end
        
function autocorrelation(spikes,Num_spikes,resolution,bandwidth,Tmax) #Just calling all the functions in a row
    h=resolution;support=[x for x in 0:h:Tmax];
    nb=length(bandwidth);bars_hist=fill(Any[],nb);x=[1,2,3,4,5];y=[1,2,3,4,5];line_hist=[Spline1D(x, y)];quad_hist=[Spline1D(x, y)];cube_hist=[Spline1D(x, y)]
    deleteat!(line_hist,1);deleteat!(quad_hist,1);deleteat!(cube_hist,1);
    bins=binned_autocorrelation(spikes,Num_spikes,resolution,Tmax)
    support=bins[1]
    g=bins[2]
    λ=bins[3]
    
    for i in 1:nb
        nh=bandwidth[i]
        temp_hist=spline_autocorrelation(g,Num_spikes,resolution,nh,Tmax,λ)
        bars_hist[i]=temp_hist[1]
        line_hist=vcat(line_hist,temp_hist[2])
        quad_hist=vcat(quad_hist,temp_hist[3])
        cube_hist=vcat(cube_hist,temp_hist[4])
    end
    
    return support,bars_hist,line_hist,quad_hist,cube_hist,sum
end



autocorrelation (generic function with 1 method)

# Estimation of width

### Integration

### Module for integrating with Gauss-Legendre quadrature

In [4]:
function leg_re_shape(roots,weights,x1,x2) #Re-scaling the Legendre polynomial zeros for the quadrature rule
    N=length(roots)
    scaled_roots=zeros(N)
    scaled_weights=zeros(N)
    for i in 1:N
        scaled_roots[i]=(x1+x2)/2+roots[i]*(x2-x1)/2
    end
    scaled_weights=(x2-x1)/2 .*weights
    return scaled_roots, scaled_weights
end
function quadrature_integral(f,domf,lim_inf,lim_sup,n_nodes,argument)#argument syntax x->f(x)
    x1=lim_inf;x2=lim_sup
    nodes,weights = gausslegendre(n_nodes)
    nodes,weights=leg_re_shape(nodes,weights,x1,x2)
    f=[my_interpolate(f,domf,x0) for x0 in nodes]
    f=argument.(f)
    integral=dot(f,weights)
    return integral
end

quadrature_integral (generic function with 1 method)

### Variance, mean and Fano factor

In [5]:
function hawkes_area(spikes,Num_spikes,Tmax) #Computation of the normalization
    Spikes=[x for x in spikes];N=length(spikes);T=spikes[end]-spikes[1];λ=lambda_bar(spikes);
    n=0

    for i in 1:N
        temp_diff=Spikes.-Spikes[1]
        k=searchsortedlast(temp_diff,Tmax)
        deleteat!(Spikes,1)
        n+=k
    end
    
    return n
end

function hawkes_variance(spikes,Num_spikes,Tmax,remove)#Computation of the sum of squares
    Spikes=[spikes[i] for i in 1:Num_spikes];N=length(spikes);T=spikes[end]-spikes[1];λ=lambda_bar(spikes);
    var=0;n=0
    for i in 1:Num_spikes
        temp_diff=Spikes.-Spikes[1]
        k=searchsortedlast(temp_diff,Tmax)
        si=[temp_diff[j] for j in 1:k]
        temp_squ=sum(x^2 for x in temp_diff[1:k])
        var+=sum(temp_squ)
        deleteat!(Spikes,1)
        n+=k
    end
    var=(var-remove*(Tmax)^3/(3))
    return var
end

function shorten(spikes,num_removed) #shorten a spike train to ensure stationarity
    new_spikes=spikes[num_removed+1:end]
    new_spikes=new_spikes.-spikes[num_removed]
    return new_spikes
end

function lambda_bar(spikes) #Mean rate
    T=spikes[end]
    λ=length(spikes)/T
    return λ
end

function make_bins(spikes,L) #Counting the number of spikes in bins of length L
    T=spikes[end]; m=Integer(floor(T/L)); count=zeros(m); uk=0;vk=0
    for k in 1:m
        uk=vk+1
        vk=searchsortedlast(spikes,k*L)
        count[k]=vk-uk+1
    end
    return count
end

function mean_and_variance(spikes,L) #mean and variance spike count
    #λ=lambda_bar(spikes)
    count=make_bins(spikes,L)
    m=length(count)
    λ=sum(count/L)/(m)
    σ2=(1/(m-1))*sum([(N-λ*L)^2 for N in count])
    return λ,σ2
end

function fanofactor(spikes,L) #computation of the fano factor
    mv=mean_and_variance(spikes,L)
    λL=mv[1]*L
    σ2=mv[2]
    F=σ2/λL
    return F
end

function estimate_alpha(spikes,L) #estimation of alpha from fano factor
    F=fanofactor(spikes,L)
    α=1-sqrt(1/F)
    return α,F
end

function estimate_width(spikes,Num_spikes,W) #estimation of the autocorrelation width
    spikes=spikes[1:Num_spikes]
    dots=[W,2*W]
    dotareas=[hawkes_area(spikes,Num_spikes,x) for x in dots]
    slope=(dotareas[2]-dotareas[1])/W
    ord=dotareas[1]-slope*dots[1]
    varest=hawkes_variance(spikes,Num_spikes,W,slope)
    σ=sqrt(abs(varest)/ord)
    
    return σ
end

estimate_width (generic function with 1 method)

# RKHS

In [6]:
function re_shape(T,α) #Takes a value between -1 and 1 and translates it to a value between 0 and infinity
    N=length(T);Nc=Integer((N+1)/2)
    τ=zeros(N)
    for i in 1:N
        τ[i]=α*(1-T[i])/(1+T[i])
    end
    return τ
end
function repkernel(x,y)#Reproducing Kernel of the Hilbert space
    if y<=x
        K=7/3+3x/2+(2x+3/2)y+(x*y^2)/2-(y^3)/6
    else
        K=7/3-(x^3)/6+3x/2+((x^2)/2+2x+3/2)y
    end
    return K
end
function my_interpolate(f,domf,x0)#Interpolate a given function at the given value of x0
    if x0>domf[1]&&x0<domf[end]
            a=searchsortedlast(domf,x0)
            ya=f[a];yb=f[a+1];xa=domf[a];xb=domf[a+1]
            y=ya+(yb-ya)*((x0-xa)/(xb-xa))
        else
            y=0.0
        end

    return y
end

function my_interpolate(f::Spline1D,domf,x0) #Interpolate spline type functions
    if abs(x0)<=domf[end]
        if x0>=0
            y=evaluate(f,x0)
        else
            y=evaluate(f,-1*x0)
        end
    else
        y=0.0
    end
    return y
end
function my_integral(f,li,ls,bins,xi,x,Φ,domΦ)#Numerical Integral
    h=(ls-li)/bins
    grid=[k for k in li:h:ls]
    dim=length(grid)
    Sn=0
    for i in 2:dim-1
        Si=f(x,xi,grid[i],Φ,domΦ)+4*f(x,xi,(grid[i]+grid[i+1])/2,Φ,domΦ)+f(x,xi,grid[i+1],Φ,domΦ)
        Sn=Sn+Si
    end
    Sn=(h/6)*Sn
    return Sn
end
function K(x,xi,y,Φ,domΦ)#Kernel of the integral equation under the change of variable
    arg=re_shape(xi,α)[1]-re_shape(y,α)[1]
    K=my_interpolate(Φ,domΦ,arg)
    return K
end
function KR(x,xi,y,Φ,domΦ)#Kernel of the integral equation times the reproducing kernel under the change of variable
    arg=re_shape(xi,α)[1]-re_shape(y,α)[1]
    K=my_interpolate(Φ,domΦ,arg)
    R=repkernel(x,y)-repkernel(xi,x)
    KR=K*R
    return KR
    end
function G(x,Φg,domΦ)#Funtion g in the integral equation under the change of variable
    arg=re_shape(x,α)[1]
    G=my_interpolate(Φg,domΦ,arg)
    return G
end
function ψ(domψ,dense,Φ,domΦ)#Generation of a complete system in the Hilbert space
    L=length(domψ)
    ψtemp=zeros(L)
    N=length(dense)
    ψ=Dict()
    for i in 1:N
        xi=dense[i]
        for j in 1:L
            x=domψ[j]
            ψtemp[j]=((xi+1)^2+2α*my_integral(K,-1,1,bins,xi,x,Φ,domΦ))*repkernel(xi,x)+2α*my_integral(KR,-1,1,bins,xi,x,Φ,domΦ)
        end
        ψi=Dict("ψ$i"=>ψtemp)
        ψ=merge(ψ,ψi)
        ψtemp=zeros(L)
    end
    return ψ
end
function my_fixderivative(f,domf)#Numerical first derivative in x=-1
    d=(-3*f[1]+4*f[2]-f[3])/(2*(domf[2]-domf[1]))
    return d
end
function my_secderivative(f,domf,j)#Numerical second derivative at any point
    N=length(domf)
    if j==1
        d=(2*f[1]-5*f[2]+4*f[3]-f[4])/((domf[2]-domf[1])^2)
    end
    if j==N
        d=(2*f[N]-5*f[N-1]+4*f[N-2]-f[N-3])/((domf[2]-domf[1])^2)
    end
    if 1<j<N
        d=(f[j+1]-2f[j]+f[j-1])/(domf[2]-domf[1])^2
    end
    return d
end
function inner_product(f,g,domf)#Inner product as defined in this hilbert space
    N=length(domf)
    h=domf[2]-domf[1]
    derf=[my_secderivative(f,domf,j) for j in 1:N]
    derg=[my_secderivative(g,domf,j) for j in 1:N]
    fg=derf.*derg
    Sn=0
    for i in 1:N-1
        x0=(domf[i+1]+domf[i])/2
        Si=fg[i]+4*my_interpolate(fg,domf,x0)+fg[i+1]
        Sn=Sn+Si
    end
    Sn=(h/6)*Sn
    dot=f[1]*g[1]+my_fixderivative(f,domf)*my_fixderivative(g,domf)+Sn
    return dot
end
function project(f,g,domf)#Orthogonal projection as given by the inner product
    a=inner_product(f,g,domf)/inner_product(g,g,domf)
    proj=a*g
    return proj
end
function grammsch(ψ,domψ,dense)#Gramm-Schmidt orthonormalization
    N=length(dense)
    L=length(domψ)#orthogonalization
    ψ1=ψ["ψ1"]
    Ψtemp1=ψ1
    Ψtemp=Dict("Ψtemp1"=>Ψtemp1)
    Ψ=Dict()
    for i in 2:N
        w=ψ["ψ$i"]
        for j in 1:i-1
            u=Ψtemp["Ψtemp$j"]
            w=w-project(w,u,domψ)
        end
        Ψtempi=Dict("Ψtemp$i"=>w)
        Ψtemp=merge(Ψtemp,Ψtempi)
    end
    #Now, normalization
    for i in 1:N
        a=Ψtemp["Ψtemp$i"]
        a=a/sqrt(inner_product(a,a,domψ))
        Ψnorm=Dict("Ψ$i"=>a)
        Ψ=merge(Ψ,Ψnorm)
    end
    return Ψ
end
function Bs(ψ,Ψ,domψ,dense)#Fourier coefficients for the ψ
    N=length(dense)
    Bij=Dict()
    for i in 1:N
        for j in 1:i
            u=inner_product(ψ["ψ$i"],Ψ["Ψ$j"],domψ)
            Btemp=Dict("B$i$j"=>u)
            Bij=merge(Bij,Btemp)
        end
    end
    return Bij
end
function βs(B,dense)#Change of basis coefficients
    N=length(dense)
    βij=Dict()
    for i in 1:N
        #terms in the diagonal
        u=1/B["B$i$i"]
        βtemp=Dict("β$i$i"=>u)
        βij=merge(βij,βtemp)
        for j in 1:i-1
            #terms outside the diagonal
            v=βij["β$i$i"]*(-sum(B["B$i$k"]*βij["β$k$j"] for k in j:i-1))
            βtemp=Dict("β$i$j"=>v)
            βij=merge(βij,βtemp)
        end
    end
    return βij
end
function solution(β,domψ,Ψ,dense,Φg,domΦ)#Solution to the integral equation
    N=length(dense);L=length(domψ)    
    u=sum(sum(β["β$i$k"]*G(dense[k],Φg,domΦ)*Ψ["Ψ$i"] for k in 1:i) for i in 1:N)
    
    for l in 1:L
        u[l]=(1+domψ[l])^2*u[l]
    end
    return u
end

function rkhs_estimation(domψ,dense,Φ,Φg,domΦ) #This function does a kernel estimation solving a Wiener-Hopf equation
    ψi=ψ(domψ,dense,Φ,domΦ)
    Ψ=grammsch(ψi,domψ,dense)
    B=Bs(ψi,Ψ,domψ,dense)
    β=βs(B,dense)
    u=solution(β,domψ,Ψ,dense,Φg,domΦ)
    return u
end

function Hawkes_RKHS(spikes,Num_spikes, resolution, bandwidth, Tmax,spline_order) #All functions integrated into one
    hist=autocorrelation(spikes, Num_spikes, resolution, bandwidth, Tmax);
    domΦ=[x for x in hist[1]]
    r=Integer(spline_order+2)
    histogram=hist[r][1]
    histogram2=hist[r][1]
    if r==2
        domΦ=vcat(-1*reverse(domΦ[2:end]),domΦ)
        histogram2=vcat(reverse(histogram[2:end]),histogram)
    end
    Φ=histogram2
    Φg=histogram2
    domψ=[x for x in -1:dt:1]
    dommod=re_shape(domψ,α);
    dense=[cos(n) for n in 0:20];
    kernel=rkhs_estimation(domψ,dense,Φ,Φg,domΦ)
    dommod=reverse(dommod);kernel=reverse(kernel)
    
    return hist[1],histogram,dommod,kernel
end

Hawkes_RKHS (generic function with 1 method)

# Optimal bin size

In [7]:
function empirical_binsize(N,σ,spline_order)#Empirical law for bin-size estimation
    if spline_order==0
        h=8.05*σ*N^(-1/3)
    end
    if spline_order==1
        h=0.04*σ*N^(-1/5)
    end
    if spline_order==2
        h=0.12*σ*N^(-1/7)
    end
    if spline_order==3
        h=0.19*σ*N^(-1/9)
    end
    
    return h
end

empirical_binsize (generic function with 1 method)

# ISI to spikes

In [8]:
function make_spikes(ISI)#Create a spike train from ISI data
    N=length(ISI)+1;t=0;spikes=zeros(N)
    for i in 2:N
        t+=ISI[i-1]
        spikes[i]=t
    end
    return spikes[2:end]
end 
function periodic_boundary(spikes) #Periodic boundary conditions (not a good idea)
    periodic_spikes=vcat(spikes,spikes)
    return periodic_spikes
end

periodic_boundary (generic function with 1 method)

# These parameters are global

In [1]:
dt=0.005   #resolution between the points in the [-1,1] interval for the RKHS method
bins=500;  #number of bins for the integral with parallelogram method in the RKHS method(Could have used the legendre quadrature, but I never implemented it)
α=200;     #Shape factor for the transformation between the interval [-1,1] to [0,∞) 
#For estimation on the interval [0,T] it is recommended to use α=0.7*T

# Estimation real data

In [10]:
#Example of the method applied to real data
nameshomotor=readdir("HOMotor")
len_homotor=length(nameshomotor)
homotor_hist=Dict()
homotor_kers=Dict()
homotor_alfa=Dict()
save("HomotorHist.jld",homotor_hist)
save("HomotorKers.jld",homotor_kers)
save("HomotorAlfa.jld",homotor_alfa)
fanorange=[x for x in 0:10:8000]
fanorange[1]=fanorange[1]+1
@time for i in 1:len_homotor
    homotor_file=open(string("HOMotor//",nameshomotor[i]))
    homotor_lines=readlines(homotor_file)
    ISIs=parse.(Float64,homotor_lines);
    spikes=make_spikes(ISIs)
    fano=[estimate_alpha(spikes,L)[1] for L in fanorange]
    αf=mean(fano[200:end])
    sol0=Hawkes_RKHS(spikes,2000,0.1,200, 800,0)
    α0=quadrature_integral(sol0[4], sol0[3], sol0[3][1], sol0[3][end-1],1000, x->x)
    sol1=Hawkes_RKHS(spikes,2000,0.1,250, 800,1)
    α1=quadrature_integral(sol1[4], sol1[3], sol1[3][1], sol1[3][end-1],1000, x->x)
    sol2=Hawkes_RKHS(spikes,2000,0.1,200, 800,2)
    α2=quadrature_integral(sol2[4], sol2[3], sol2[3][1], sol2[3][end-1],1000, x->x)
    sol3=Hawkes_RKHS(spikes,2000,0.1,200, 800,3)
    α3=quadrature_integral(sol3[4], sol3[3], sol3[3][1], sol3[3][end-1],1000, x->x)
    histograms=[sol0[1],sol0[2],evaluate(sol1[2],sol0[1]),evaluate(sol2[2],sol0[1]),evaluate(sol3[2],sol0[1])]
    kernels=[sol0[3],sol0[4],sol1[4],sol2[4],sol3[4]]
    αs=[αf,α0,α1,α2,α3]
    temp_histograms=Dict("SpikeTrain$i"=>histograms)
    temp_kernels=Dict("SpikeTrain$i"=>kernels)
    temp_alfa=Dict("SpikeTrain$i"=>αs)
    print("SpikeTrain$i ")
    homotor_hist=load("HomotorHist.jld")
    homotor_hist=merge(homotor_hist,temp_histograms)
    save("HomotorHist.jld",homotor_hist)
    homotor_kers=load("HomotorKers.jld")
    homotor_kers=merge(homotor_kers,temp_kernels)
    save("HomotorKers.jld",homotor_kers)
    homotor_alfa=load("HomotorAlfa.jld")
    homotor_alfa=merge(homotor_alfa,temp_alfa)
    save("HomotorAlfa.jld",homotor_alfa)
    close(homotor_file)
end

    
    

SpikeTrain1 SpikeTrain2 SpikeTrain3 SpikeTrain4 SpikeTrain5 SpikeTrain6 SpikeTrain7 SpikeTrain8 SpikeTrain9 SpikeTrain10 SpikeTrain11 SpikeTrain12 SpikeTrain13 SpikeTrain14 SpikeTrain15 SpikeTrain16 SpikeTrain17 SpikeTrain18 SpikeTrain19 SpikeTrain20 SpikeTrain21 SpikeTrain22 SpikeTrain23 SpikeTrain24 SpikeTrain25 SpikeTrain26 SpikeTrain27 SpikeTrain28 SpikeTrain29 SpikeTrain30 SpikeTrain31 SpikeTrain32 SpikeTrain33 SpikeTrain34 SpikeTrain35 SpikeTrain36 SpikeTrain37 SpikeTrain38 SpikeTrain39 SpikeTrain40 SpikeTrain41 SpikeTrain42 SpikeTrain43 SpikeTrain44 SpikeTrain45 SpikeTrain46 SpikeTrain47 SpikeTrain48 SpikeTrain49 SpikeTrain50 SpikeTrain51 SpikeTrain52 SpikeTrain53 SpikeTrain54 SpikeTrain55 SpikeTrain56 SpikeTrain57 SpikeTrain58 SpikeTrain59 SpikeTrain60 SpikeTrain61 SpikeTrain62 SpikeTrain63 SpikeTrain64 SpikeTrain65 SpikeTrain66 SpikeTrain67 SpikeTrain68 SpikeTrain69 SpikeTrain70 SpikeTrain71 SpikeTrain72 SpikeTrain73 SpikeTrain74 SpikeTrain75 SpikeTrain76 SpikeTrain77 SpikeTra